# RAASTA Pass 4 (fast mix, 8000 photos)

**Connect T4 first. Mix and Train must be the same runtime.**

Do not mix on CPU and then switch to GPU. `/content/balanced_mix` is deleted when the runtime changes. Do not Connect without GPU.

Free Colab dies on ~21k Drive photos (scan alone ~1 hour). This notebook **caps the mix at 8000** (all rare classes first, then fill with potholes).

| Step | Cell | Time |
|------|------|------|
| 1 | Setup + pip (must print GPU True T4) | 2 min |
| 2 | Mix collect (pickle if `candidates_pass4.pkl` exists) | 1 min / 10–25 min |
| 3 | Mix write (8000 symlinks) | 3–10 min |
| 4 | Train immediately | 1.5–3 hours |
| 5 | Export after `New best` | 5 min |

Skip ExDark. Skip person download if `open_images_person` is on Drive.

Start from Pass 3: `weights/m2_m5_night_public_v1/finetune/weights/best.pt`

Keep the tab in front until Train is actually running epochs.

In [ ]:
from google.colab import drive
from pathlib import Path
import torch

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/raasta')
EXTRACTED = ROOT / 'extracted'
MERGED = ROOT / 'merged'
WEIGHTS = ROOT / 'weights'
EXPORTS = ROOT / 'exports'
RAW = ROOT / 'raw'
CACHE = RAW / 'candidates_pass4.pkl'
for p in (EXTRACTED, MERGED, WEIGHTS, EXPORTS, RAW):
    p.mkdir(parents=True, exist_ok=True)

BAL_W = WEIGHTS / 'm2_m5_balanced_v1/finetune/weights'
print('Drive', ROOT)
print('GPU', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — Connect T4, do not continue')
print('merged', (MERGED / 'images' / 'train').exists())
print('person OI', (EXTRACTED / 'open_images_person').exists())
print('mix cache', CACHE.exists(), CACHE)
print('pass3 best', (WEIGHTS / 'm2_m5_night_public_v1/finetune/weights/best.pt').exists())
print('last.pt', (BAL_W / 'last.pt').exists())
print('best.pt pass4', (BAL_W / 'best.pt').exists())
print('mix on this VM', (Path('/content/balanced_mix') / 'data.yaml').exists())
assert torch.cuda.is_available(), 'Runtime → Change runtime type → T4 GPU → Connect. Then rerun this cell.'

In [ ]:
!pip -q install ultralytics pyyaml
print('packages ready')

## Mix collect (fast)

If `candidates_pass4.pkl` exists, this loads in under a minute. Otherwise it scans Drive folders (no ExDark, no Image.open).

In [ ]:
from pathlib import Path
import pickle, time

ROOT = Path('/content/drive/MyDrive/raasta')
EXTRACTED = ROOT / 'extracted'
MERGED = ROOT / 'merged'
CACHE = ROOT / 'raw' / 'candidates_pass4.pkl'
CLASS_NAMES = [
    'pothole', 'crack', 'speed_bump', 'person', 'cow', 'buffalo',
    'dog', 'cat', 'horse', 'donkey', 'goat',
]
CLASS_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
ALIAS = {
    'pothole': 'pothole', 'd40': 'pothole', 'hole': 'pothole',
    'crack': 'crack', 'd00': 'crack', 'd10': 'crack', 'd20': 'crack',
    'speed bump': 'speed_bump', 'speedbump': 'speed_bump', 'bump': 'speed_bump',
    'person': 'person', 'people': 'person', 'pedestrian': 'person',
    'cow': 'cow', 'buffalo': 'buffalo', 'dog': 'dog', 'cat': 'cat',
    'horse': 'horse', 'donkey': 'donkey', 'mule': 'donkey', 'goat': 'goat',
}

def data_dir(name):
    for p in (Path('/content/extracted') / name, EXTRACTED / name):
        if p.exists():
            return p
    return EXTRACTED / name

def map_yolo_lines(lbl, names=None):
    lines = []
    for line in lbl.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cid = int(float(parts[0]))
        if names:
            key = ' '.join(str(names.get(cid, '')).lower().replace('_', ' ').split())
            alias = ALIAS.get(key)
            if alias is None:
                continue
            cid = CLASS_TO_ID[alias]
        elif cid >= 11:
            continue
        parts[0] = str(cid)
        lines.append(' '.join(parts[:5]))
    return lines

def yolo_pairs(root):
    pairs = []
    layouts = [
        ('train/images', 'train/labels'), ('valid/images', 'valid/labels'),
        ('val/images', 'val/labels'), ('images/train', 'labels/train'),
        ('images/val', 'labels/val'),
    ]
    if not root.exists():
        return pairs
    roots = [root] + [p for p in root.iterdir() if p.is_dir()]
    seen = set()
    for base in roots:
        for ri, rl in layouts:
            img_dir, lbl_dir = base / ri, base / rl
            if not img_dir.is_dir() or not lbl_dir.is_dir():
                continue
            for img in img_dir.iterdir():
                if img.suffix.lower() not in IMG_EXT:
                    continue
                lbl = lbl_dir / f'{img.stem}.txt'
                if lbl.exists() and img not in seen:
                    seen.add(img)
                    pairs.append((img, lbl))
    return pairs

if CACHE.exists() and CACHE.stat().st_size > 1000:
    raw = pickle.load(CACHE.open('rb'))
    candidates = [(Path(p), lines, src) for p, lines, src in raw if Path(p).exists()]
    print('Loaded cache', len(candidates), CACHE)
else:
    t0 = time.time()
    candidates = []

    def add(img, lines, src):
        if lines:
            candidates.append((img, lines, src))

    print('A) merged')
    n = 0
    if (MERGED / 'images' / 'train').exists():
        for split in ('train', 'val'):
            img_dir = MERGED / 'images' / split
            lbl_dir = MERGED / 'labels' / split
            if not img_dir.exists():
                continue
            for img in img_dir.iterdir():
                if img.suffix.lower() not in IMG_EXT:
                    continue
                lbl = lbl_dir / f'{img.stem}.txt'
                if not lbl.exists():
                    continue
                add(img, map_yolo_lines(lbl), 'old')
                n += 1
                if n % 2000 == 0:
                    print('  merged', n, f'{time.time()-t0:.0f}s')
    print('merged', n)

    print('B) pothole / animals / person')
    jobs = [
        ('pot', 'pothole_dataset_v8', {0: 'pothole'}),
        ('ani', 'animals_yolo', None),
        ('per', 'open_images_person', {3: 'person'}),
    ]
    for src, folder, names in jobs:
        pairs = yolo_pairs(data_dir(folder))
        for img, lbl in pairs:
            add(img, map_yolo_lines(lbl, names), src)
        print(src, len(pairs), f'{time.time()-t0:.0f}s')

    print('SKIP ExDark')
    print('raw candidates', len(candidates), f'{time.time()-t0:.0f}s')
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    pickle.dump([(str(i), L, s) for i, L, s in candidates], CACHE.open('wb'))
    print('saved', CACHE)

assert len(candidates) > 1000, 'Too few candidates. Check Drive folders.'
print('OK', len(candidates), 'photos in mix list')

## Mix write (8000 symlinks, not copies)

Caps at 8000 so Train can finish on free Colab. Rare classes first, then potholes fill the rest. **Start Train in this same runtime.** Do not disconnect.

In [ ]:
import os, random, shutil
from pathlib import Path
from collections import Counter, defaultdict

CLASS_NAMES = [
    'pothole', 'crack', 'speed_bump', 'person', 'cow', 'buffalo',
    'dog', 'cat', 'horse', 'donkey', 'goat',
]
OUT = Path('/content/balanced_mix')
MAX_IMAGES = 8000
PER_CLASS = 700

if OUT.exists():
    shutil.rmtree(OUT)
for split in ('train', 'val'):
    (OUT / 'images' / split).mkdir(parents=True)
    (OUT / 'labels' / split).mkdir(parents=True)

assert 'candidates' in dir() and len(candidates) > 1000, 'Run mix collect first'

rng = random.Random(42)
rng.shuffle(candidates)

by = defaultdict(list)
for img, lines, src in candidates:
    ids = {int(x.split()[0]) for x in lines if x.split()}
    for i in ids:
        if 0 <= i < len(CLASS_NAMES):
            by[CLASS_NAMES[i]].append((img, lines, src))

picked, seen = [], set()

def take(name, n):
    count = 0
    for item in by.get(name, []):
        k = str(item[0])
        if k in seen:
            continue
        seen.add(k)
        picked.append(item)
        count += 1
        if count >= n:
            break

for name in CLASS_NAMES:
    take(name, PER_CLASS)
for item in candidates:
    if len(picked) >= MAX_IMAGES:
        break
    k = str(item[0])
    if k not in seen:
        seen.add(k)
        picked.append(item)

print('using', len(picked), 'of', len(candidates))

box_counts = Counter()
file_counts = {'train': 0, 'val': 0}

def write_sample(img, lines, split, prefix):
    if not img.exists():
        return
    stem = f'{prefix}_{file_counts[split]}_{img.stem}'[:180]
    dst = OUT / 'images' / split / f'{stem}{img.suffix.lower()}'
    try:
        os.symlink(img, dst)
    except FileExistsError:
        pass
    (OUT / 'labels' / split / f'{stem}.txt').write_text('\n'.join(lines) + '\n')
    file_counts[split] += 1
    for line in lines:
        box_counts[CLASS_NAMES[int(line.split()[0])]] += 1

for img, lines, src in picked:
    split = 'val' if rng.random() < 0.12 else 'train'
    write_sample(img, lines, split, src)

(OUT / 'data.yaml').write_text(
    f'path: {OUT}\ntrain: images/train\nval: images/val\nnames:\n'
    + ''.join(f'  {i}: {n}\n' for i, n in enumerate(CLASS_NAMES))
)
print('---- FINAL ----')
print('Files', dict(file_counts))
print('Boxes', dict(box_counts))
print(OUT / 'data.yaml')
assert file_counts['train'] > 200
print('Start Train next. Keep this runtime.')

## Train (this is the long GPU step)

Run **immediately** after mix write. Keep the tab in front. If CUDA OOM, set `batch=4` and rerun this cell only (mix is still on `/content`).

In [ ]:
from pathlib import Path
import gc, torch
from ultralytics import YOLO

OUT = Path('/content/balanced_mix')
WEIGHTS = Path('/content/drive/MyDrive/raasta/weights')
assert (OUT / 'data.yaml').exists(), 'Run mix write first in THIS runtime'

CANDIDATES = [
    WEIGHTS / 'm2_m5_night_public_v1/finetune/weights/best.pt',
    WEIGHTS / 'm2_m5_people_animals_v1/finetune/weights/best.pt',
]
BEST = next((p for p in CANDIDATES if p.exists()), None)
if BEST is None:
    found = sorted(WEIGHTS.rglob('best.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
    BEST = found[0] if found else None
assert BEST is not None, BEST
print('Starting from', BEST)

gc.collect()
assert torch.cuda.is_available(), 'Connect T4 GPU before Train'
torch.cuda.empty_cache()
print('GPU', torch.cuda.get_device_name(0))

RUN = WEIGHTS / 'm2_m5_balanced_v1'
RUN.mkdir(parents=True, exist_ok=True)
model = YOLO(str(BEST))
model.train(
    data=str(OUT / 'data.yaml'),
    epochs=30,
    imgsz=640,
    batch=8,
    workers=2,
    device=0,
    project=str(RUN),
    name='finetune',
    exist_ok=True,
    pretrained=True,
    patience=12,
    cache=False,
    amp=True,
    plots=True,
    save_period=5,
    mosaic=0.7,
    close_mosaic=8,
    copy_paste=0.15,
    fliplr=0.5,
    degrees=8.0,
    translate=0.1,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.5,
)
print('New best', RUN / 'finetune/weights/best.pt')

### Resume if Train died halfway

Reconnect **T4** first.

- `last.pt` True **and** mix True: uncomment Resume below.
- `last.pt` True **but mix gone**: rerun mix collect + mix write, then Resume.
- `last.pt` False: do not Resume. Mix + Train from the start.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

last = Path('/content/drive/MyDrive/raasta/weights/m2_m5_balanced_v1/finetune/weights/last.pt')
print('last.pt exists', last.exists(), last)
# Uncomment only to resume:
# YOLO(str(last)).train(resume=True)

## Export phone file

Run only after Train prints `New best`.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import shutil

ROOT = Path('/content/drive/MyDrive/raasta')
FT = ROOT / 'weights/m2_m5_balanced_v1/finetune/weights/best.pt'
EXPORTS = ROOT / 'exports'
EXPORTS.mkdir(parents=True, exist_ok=True)
assert FT.exists(), FT
model = YOLO(str(FT))
export_path = model.export(format='tflite', imgsz=640, int8=False)
shutil.copy2(FT, EXPORTS / 'raasta_m2_m5_balanced.pt')
shutil.copy2(export_path, EXPORTS / 'raasta_m2_m5_balanced.tflite')
print('Download', EXPORTS / 'raasta_m2_m5_balanced.tflite')
print('Replace E:\\Android\\projects\\raasta_app\\assets\\models\\raasta_m2_m5.tflite')